In [1]:
%pip install -q cassio datasets tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install PyPDF2


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_astradb import AstraDBVectorStore
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain,create_history_aware_retriever

In [ ]:
# --- 1. CONFIGURATION & FIXES ---
# Fix for Python 3.12+ Cassandra driver compatibility on Windows
os.environ['CASSANDRA_DRIVER_DEFAULT_CONNECTION_CLASS'] = 'AsyncioConnection'
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [8]:
# --- 2. DATA INGESTION
pdf_reader = PdfReader("budget_speech.pdf")

raw_text = ''
for pages in pdf_reader.pages:
    content = pages.extract_text()
    if content:
        raw_text += content

In [10]:
print(raw_text)

1 
  
 
 
 
 
 
 
BUDGET SPEECH  
2025-26 
 
 
 
 
 
 
 
 2 
 Respected Speaker Sir! I am presenting the budget for the 
financial year 2025-26 in this Hon’ble House. Today is a historic 
day. This is not an ordinary budget. The people of Delhi and the 
entire country are seeing today that the new government of Delhi, 
which has been elected with a historic mandate, has been voted by 
the people of Delhi with great hopes and expectations. How will the 
first budget of that government be? I want to tell you that this budget 
is not just an account of government income and expenditure but 
is the first resolved step taken towards the development of Delhi 
which has become miserable and worse in the last ten years. A 
Delhi that becomes a confluence of glorious history and bright 
future. 
 
2. I and my government, while paying obeisance to the people of 
Delhi and Mother Yamuna, accept this responsibility with all 
humility and pledge to fulfill it properly. 
 
3. This budget of ours is 

In [11]:
# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200,
)
texts = text_splitter.split_text(raw_text)

In [12]:
texts[:50]

['1 \n  \n \n \n \n \n \n \nBUDGET SPEECH  \n2025-26 \n \n \n \n \n \n \n \n 2 \n Respected Speaker Sir! I am presenting the budget for the \nfinancial year 2025-26 in this Hon’ble House. Today is a historic \nday. This is not an ordinary budget. The people of Delhi and the \nentire country are seeing today that the new government of Delhi, \nwhich has been elected with a historic mandate, has been voted by \nthe people of Delhi with great hopes and expectations. How will the \nfirst budget of that government be? I want to tell you that this budget \nis not just an account of government income and expenditure but \nis the first resolved step taken towards the development of Delhi \nwhich has become miserable and worse in the last ten years. A \nDelhi that becomes a confluence of glorious history and bright \nfuture.',
 "which has become miserable and worse in the last ten years. A \nDelhi that becomes a confluence of glorious history and bright \nfuture. \n \n2. I and my government, wh

In [29]:
# --- 3. INITIALIZE MODERN 2026 COMPONENTS ---
# Use Ollama for free local embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Use Groq for fast, low-cost LLM
llm = ChatGroq(model="llama-3.1-8b-instant")

In [26]:
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F1900C19A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F19071A060>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [32]:
# Initialize AstraDB Vector Store (Replaces legacy 'Cassandra' class)
vector_store = AstraDBVectorStore(
    collection_name="pdf_query_2026",
    embedding=embeddings,
    token=ASTRA_DB_APPLICATION_TOKEN,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
)

# Add texts to the vector store
vector_store.add_texts(texts[:50])

['cd9c557f841447b09c9e1b8de88747f9',
 'd603e9638e9444309443b45033099fc8',
 '1b514a5a8b904c9c91b045596e678c02',
 '296977eee6d9487ea287b19997eebd97',
 '10af010adb6b4908909755a37ed4b05e',
 '7b76b001cde7445eab5b6641574547cf',
 'fc504c6863614979981a913c5f6ac665',
 '64626c8100284890a97d9b7d3f31e548',
 'ce394f66debe4251a8e2d2fc799b7c9c',
 'a963a77bf57b4f1295218440955faf6c',
 '4420c51ad92649dab89f4cebe58a2384',
 '4e7adc2927704d549f885f6c4923b416',
 'b0f6ff5422fd4983af4e74b548a16394',
 '1486b3f9c3cb499ca6435affee70881b',
 '0518d9cafa0241fda382301fce440c64',
 'be6c41bf05404e3a999d83713544fa6e',
 '45451ecb01f84e3585e4191280d3338a',
 '3c8dfa17b592433a8edbf7af5cc65c59',
 '0cb78fb055654f669149f4092d290e19',
 '2142795f6476487eae5a0e455c7e248f',
 'c999d73ce13e49a0aa01f3caaa422bb6',
 '89bef39b550b46fa99708199f5e4408d',
 '417d829ab1e64e37821547d5991d2aa2',
 '7918f43d5b7a46839955826dd16b41b8',
 '5fbe9319ce6341e887ce5c2bd125757c',
 'adacc0814b224fdba69ad6abc4a5214d',
 '58a493718da247e883d4a452df8035bb',
 

In [33]:
# --- 4. CREATE THE RETRIEVAL CHAIN (Modern Alternative to VectorStoreIndexWrapper) ---
prompt = ChatPromptTemplate.from_template("""
Answer the following question based only on the provided context:
<context>
{context}
</context>
Question: {input}""")

document_chain = create_stuff_documents_chain(llm, prompt)
retriever = vector_store.as_retriever()
rag_chain = create_retrieval_chain(retriever, document_chain)

# --- 5. RUN THE QA LOOP ---
while True:
    query_text = input("\nEnter your question (or 'quit' to exit): ").strip()
    if query_text.lower() == "quit":
        break
    
    response = rag_chain.invoke({"input": query_text})
    print(f"\nANSWER: {response['answer']}")


ANSWER: The context does not provide the current GDP rate. It only mentions that the GDP rate was lower than the entire country.

ANSWER: There is no mention of an increase in the agriculture target in the provided context. The text mentions various schemes, infrastructure development, and allocation of funds for different purposes, but it does not provide any information about an increase in the agriculture target.

ANSWER: Based on the provided context, the details about infrastructure development are as follows:

1. **Allocation of ₹1000 crore**: This amount has been allocated to improve the connectivity of Delhi with the NCR region with the support of the Central Government.

2. **Use of funds**: The funds will be used from the Central Roads Fund (CRF) of the Ministry of Road Transport and Highways (Government of India) and Urban Development Fund (UDF) of the Ministry of Urban Development (Government of India) etc.

3. **Infrastructure projects**: The funds will be used to launch 

🛠️ Key 2026 Updates Explained:
AstraDBVectorStore: Replaces the generic Cassandra class. It uses a modern HTTP API that bypasses the complex driver issues seen in the previous implementation.

AsyncioConnection Fix: This environment variable is strictly required for Python 3.12+ on Windows to prevent the DependencyException regarding the missing asyncore module.

Retrieval Chain Pattern: Instead of the legacy VectorStoreIndexWrapper, we now use create_retrieval_chain. This is the standard 2026 architecture for building production-grade RAG pipelines.

Hybrid Costs: By using Ollama locally, your embedding process is entirely free. Only the final summary synthesis uses Groq's high-speed infrastructure.

Need to Run ollam locally.